# Evaluación — validación propia (Colab, GPU T4)

Evalúa `solucion.generar_sql` sobre los 2 esquemas de validación (`datos/validacion/`, 40 preguntas)
con `scripts/evaluacion/evaluar.py`, que imita al evaluador del docente: exactitud por ejecución,
EX ponderada (fácil 1, media 2, difícil 3), límite de 30 s y máximo 3 llamadas a `ejecutar`.

1. **Baseline** (modelo base, mismo prompt), **LoRA A** y **LoRA B**, con `ejecutar` (autocorrección).
2. **Experimento de inferencia:** la mejor configuración **sin** `ejecutar`.

Los resultados se acumulan en `resultados/validacion_propia.csv` y se suben al repo.

Antes de empezar: *Entorno de ejecución → Cambiar tipo → T4 GPU* y el secreto `GH_TOKEN` en *Secrets*.
MySQL 8.4.10 se instala dentro de Colab (no hace falta Drive).

## Parámetros

In [ ]:
CONFIGS = ['baseline', 'lora_A', 'lora_B']   # archivos de configs/ que se evalúan con ejecutar
EXPERIMENTO = 'lora_B'        # la mejor configuración, para evaluarla también sin ejecutar
GIT_NOMBRE = 'AleWWH1104'     # usuario de GitHub de quien corre el notebook
GIT_CORREO = 'irisale.ayala@gmail.com'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:   # la extensión de Colab para VS Code no puede leer los Secrets
    token = getpass('Token de GitHub (github_pat_...): ')
if not os.path.exists('/content/proyecto3-deep'):
    !git clone -q https://{token}@github.com/Qu3zada22/proyecto3-deep.git /content/proyecto3-deep
%cd /content/proyecto3-deep
!git pull -q
!pip install -q -r requirements.txt
!git config user.name "{GIT_NOMBRE}" && git config user.email "{GIT_CORREO}"

## MySQL 8.4.10 y esquemas de validación

In [ ]:
!bash scripts/evaluacion/instalar_mysql_colab.sh
!python scripts/evaluacion/cargar_validacion.py
!python scripts/evaluacion/verificar_fuga.py
!python scripts/evaluacion/evaluar.py --oraculo | tail -3   # prueba del evaluador: debe dar EX 1.000

## Baseline, LoRA A y LoRA B (con ejecutar)

In [ ]:
import json
for c in CONFIGS:
    cfg = json.load(open(f'configs/{c}.json'))
    if cfg['usar_adapter'] and not os.path.exists(cfg['adapter']):
        print(f'{c}: falta {cfg["adapter"]} (todavía no se ha entrenado/subido), se omite')
        continue
    !T2S_CONFIG=configs/{c}.json python scripts/evaluacion/evaluar.py

## Experimento de inferencia: sin ejecutar (sin autocorrección)

In [ ]:
!T2S_CONFIG=configs/{EXPERIMENTO}.json python scripts/evaluacion/evaluar.py --sin-ejecutar

## Resumen y subida al repo

In [ ]:
import pandas as pd
df = pd.read_csv('resultados/validacion_propia.csv')
df['puntos'] = df.peso * df.acierto
resumen = df.groupby('config').agg(EX=('puntos', 'sum'), peso=('peso', 'sum'),
                                   aciertos=('acierto', 'sum'), tiempo_max=('tiempo_s', 'max'))
resumen['EX'] = (resumen.EX / resumen.peso).round(3)
display(resumen.drop(columns='peso'))
display(df[df.acierto == 0].groupby(['config', 'categoria_error']).size().unstack(fill_value=0))

In [ ]:
!git add resultados/validacion_propia.csv && git commit -q -m 'Resultados de la validación propia (Colab T4)'
!git pull -q --rebase && git push -q